In [ ]:
using Gridap        
using GridapGmsh
using Gridap.Geometry
using Gridap.TensorValues
using Plots
using LinearAlgebra
using  Gridap.Fields
using  Gridap.CellData
using  Gridap.ReferenceFEs  
using  Gridap.Fields

In [ ]:
model = GmshDiscreteModel("SingleEdgeNotch.msh")
writevtk(model,"SingleEdgeNotch")

In [ ]:
const ν = 0.3
const E = 210000
const G = E/(2*(1+ν))

In [ ]:
const λ = (E*ν)/((1+ν)*(1-2*ν))
const μ = G
const κ = λ + μ

In [ ]:
D = 2
G0 = 3e8
r = 0.01
τ = 1e-6 
R = 0.01
fₜ = 2700 
delT1 = 1e-4

In [ ]:
σ(ε)= λ*tr(ε)*one(ε) + 2*μ*ε

In [ ]:
degree = 2
Ω = Triangulation(model)
dΩ = Measure(Ω,degree)

In [ ]:
labels = get_face_labeling(model)

In [ ]:
Gr = get_grid(model)
nodes = get_node_coordinates(Gr)
Nₑ, Nₙ = num_cells(model), num_nodes(model)
nodeCoordX, nodeCoordY = [nodes[i][1] for i in 1:Nₙ], [nodes[i][2] for i in 1:Nₙ]
elem = get_cell_node_ids(Gr)

In [ ]:
function σ_eq(ε)
    εArray = get_array(ε)
    Λ, P = eigen(εArray)
    Λpos = diagm(0 => max.(0, Λ))
    εPos = TensorValue(P * Λpos * P')
    ψPos = 0.5 * ((tr(ε) >= 0) * (λ * tr(ε) * tr(ε)) + 2μ * (εPos ⊙ εPos))
    return √(2*ψPos*E)
end

In [ ]:
function G_kill(σ_eq)
 G_kill = 0.5*G0.*(( (σ_eq./fₜ -1)+ abs∘(σ_eq./fₜ - 1)))
    return G_kill
end

In [ ]:
function new_EnergyState(ψPlusPrev_in,ψhPos_in)
    ψPlus_in = ψhPos_in
    if ψPlus_in <= ψPlusPrev_in
        ψPlus_out = ψPlusPrev_in 
        
        damaged = false
    else
        ψPlus_out = ψPlus_in
        
        damaged = true
    end
    damaged, ψPlus_out   
end

In [ ]:
function barw_closed_form(D, delT1,m_p,G_k_in,σ_p)
    s = Cal_Var(σ_p,G_k_in,τ)
    # μ, g =  cal_mean_main(nodes, idxs, m_p, G_k_in, τ, delT, D, R)
    μ = Cal_mean(m_p,G_k_in,τ,delT1,D,σ_p)
    λc = 1
    bar_w_nds = 1.0 ./ sqrt.(1.0 .+λc .* s) .* (exp.( - (λc .* μ .*μ ) ./ (2.0 * (1.0 .+ λc .* s))) )
    return FEFunction(Vphi, bar_w_nds), μ, s
end
function Cal_mean(m_p,G_k_in,τ,delT1,D,σ_p)
    g = -4 .* G_k_in .* delT1 .* D .* m_p
    H =- 4 .* G_k_in .* delT1 .* D 
    # m_new = (m_p .- τ .* D .*g) ./ (1 .+  (τ .* D .* H .+ τ .* D.*(2 .*G_k_in .* delT ) ))
     m_new = (m_p .- τ .* D .*g) ./ (1 .+  (τ .* D .* H )) 
    return m_new
end

In [ ]:
function Cal_Var(σ_p, G_k_in, τ)
    β = sqrt.(σ_p) ./ (τ * D)
    H = -4 .* G_k_in .* delT1 .* D
    c = 1 ./(2 .* τ .* D) .+ H ./ 2 
    disc = β.^2 .+ 8 .* c
    σ_new = similar(σ_p)
    pos_mask = disc .> 0
    β_pos = β[pos_mask]
    c_pos = c[pos_mask]
    y1 = (-β_pos .+ sqrt.(β_pos.^2 .+ 8 .* c_pos)) ./ 2
    y2 = (-β_pos .- sqrt.(β_pos.^2 .+ 8 .* c_pos)) ./ 2
    y_subset = max.(y1, y2)
    σ_new[pos_mask] = 1 ./ (y_subset.^2)
    neg_mask = .!pos_mask
    σ_new[neg_mask] = 1 ./ ((-β[neg_mask] ./ 2).^2)
    σ_new .= max.(σ_new, 1e-12)
    return σ_new
end


In [ ]:
using NearestNeighbors
data = zeros(2,Nₙ)
data[1,:] =nodeCoordX
data[2,:] =nodeCoordY

points = data

balltree = BallTree(data)
idxs = inrange(balltree, points, r, true)

In [ ]:
function nonLocalGk(G_k_prev)
    GkVec = evaluate(G_k_prev,x_S)
    caches = [array_cache(GkVec) for k in 1:Threads.nthreads()]
    Gk_nds = zeros(Nₙ)  
    Gk_sum = zeros(Nₙ)
    Gk_count = zeros(Int, Nₙ)
    Threads.@threads for iel in 1:Nₑ
        cache = caches[Threads.threadid()]     
        ElNdInd = elem[iel]
    val_G = getindex!(cache, GkVec, iel)
for node in ElNdInd
        Gk_sum[node] += val_G[1]        
        Gk_count[node] += 1
        end
    end
Gk_nds = Gk_sum ./ Gk_count
  Gk_nds_NL = zeros(Nₙ)
    Threads.@threads for nd_id in 1:Nₙ
        NeighHood = idxs[nd_id]
        @inbounds Gk_nds_NL[nd_id] = (sum(Gk_nds[NeighHood]) ) / (length(NeighHood))
    end
    return Gk_nds_NL
end

In [ ]:
LoadTagId = get_tag_from_name(labels,"BottomEdge")
Γ_Load = BoundaryTriangulation(model,tags = LoadTagId)
dΓ_Load = Measure(Γ_Load,degree)
n_Γ_Load = get_normal_vector(Γ_Load)

In [ ]:
reffe_Disp = ReferenceFE(lagrangian,VectorValue{2,Float64},2)
V0_Disp = TestFESpace(model,reffe_Disp;
          conformity=:H1,
          dirichlet_tags=["BottomEdge","TopEdge"],
          dirichlet_masks=[(true,true),(false,true)])


In [ ]:
reffephi  = ReferenceFE(lagrangian,Float64,1)
Vphi  = TestFESpace(model,reffephi;
          conformity=:H1)

In [ ]:
function step_disp(uApp,uh,m_p,σ_p,ϕ,G_k_cell)
uApp1(x) = VectorValue(0.0,0.0)
uApp2(x) = VectorValue(0.0,uApp)  
U_Disp = TrialFESpace(V0_Disp,[uApp1,uApp2])
σ_eq_s = σ_eq∘((ε(uh))*(ϕ+1e-8))
G_k_in = G_kill(σ_eq_s)
update_state!(new_EnergyState,G_k_cell,G_k_in)
G_k_nds = nonLocalGk(G_k_cell)
ϕ,m_p,σ_p = barw_closed_form(D,delT,m_p,G_k_nds,σ_p)
a(u,v) = ∫( ε(v) ⊙ ((σ∘((ε(u))))  .*(ϕ+1e-8)))dΩ
l(v) = 0.0
op = AffineFEOperator(a,l,U_Disp,V0_Disp)
ls = LUSolver()
solver = LinearFESolver(ls)
uh = solve(solver,op)
    return uh, ϕ, m_p, σ_p, G_k_cell, FEFunction(Vphi, G_k_nds)
end

In [ ]:
dΩ_ro = Measure(Ω,1)
x_S = get_cell_points(dΩ_ro)

In [ ]:
cd("SENT_Hessian_OT_results") 

In [ ]:
uh = zero(V0_Disp)
Tmax = 0.0065
delT = 1e-6
vApp = 1.0
count_n = 0
T = 0.0
m_p = zeros(Nₙ).+ 1e-6
σ_p = zeros(Nₙ) 
Load = Float64[]
Displacement = Float64[]
push!(Load, 0.0)
push!(Displacement, 0.0)
bh_cell = CellState(0.0,dΩ_ro)
bh_nd_nl = nonLocalGk(bh_cell)
uh_prev = zero(V0_Disp)
uh_in_FE = uh
f_new = 1.0
ϕ_prev = interpolate_everywhere(f_new,Vphi)
ϕ = interpolate_everywhere(f_new,Vphi)
Gk_FE = zero(Vphi)
innerMax = 10
tol = 1e-2
target = (0.5, 0.5)
row = findfirst(n -> norm((n[1]-target[1], n[2]-target[2])) < tol, nodes)
m_p_ct = Float64[]
σ_p_ct = Float64[]
push!(m_p_ct, m_p[row])
push!(σ_p_ct, σ_p[row])
σ_eq_s = σ_eq∘(ε(uh))
start_time = time()
G_k_cell = CellState(0.0,dΩ_ro)
while T <= Tmax
    count_n = count_n + 1
T = T + delT
uApp  = T*vApp
    print("\n Entering displacemtent step :", float(uApp))
 for inner = 1:innerMax
uh,ϕ,m_p,σ_p,G_k_cell,Gk_FE =  step_disp(uApp,uh,m_p,σ_p,ϕ,G_k_cell)
e = uh - uh_in_FE

err = sqrt(sum( ∫( e⊙e )*dΩ ))
ϕ_prev = ϕ
uh_in_FE = uh
print("\n error = ",float(err))
        if err < 1e-8
            break 
        end  
    end
Node_Force = sum(∫( n_Γ_Load ⋅ (σ∘( (ε(uh))) ).*ϕ)dΓ_Load)
push!(Load, abs(Node_Force[2]))
push!(Displacement, uApp)
push!(m_p_ct, m_p[row])
push!(σ_p_ct, σ_p[row])
 if mod(count_n,250) == 0
writevtk(Ω,"SENT_Gaussian_results_$count_n",cellfields=["disp"=>uh, "phi"=>ϕ, "G_k"=>Gk_FE, "variance"=>FEFunction(Vphi, σ_p)])   
    end
    end
end_time = time()
elapsed_time = end_time - start_time